# Sum country level mortality for all ensemble members

Using country masks from [McDuffie et al. (2021)](https://doi.org/10.5281/zenodo.4642700) total mortality for each country is calculated and all ensemble members are saved in one file

In [ ]:
import os
import xarray as xr
import numpy as np
from utils.utils import get_scenario_config
import config
from utils.utils import require_dir
import pathlib

In [ ]:
# === Path config ===
MASK_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")

in_file = "GBD_Country_Masks_0.10.nc"
in_path = os.path.join(MASK_DIR, in_file)
country_mask = xr.open_dataarray(in_path)

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]
dates = f"{years.start}-{years.stop}"

MORTALITY_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "pm25" / "gridpoint_mortality")

# === Main loop ===
ensembles = []
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")

    in_file = f"Mortality_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(MORTALITY_DIR, in_file)

    if not os.path.exists(in_path):
        print(f"Missing: {in_path}")
        continue
    da = xr.open_dataarray(in_path)

    countries = []
    # Loop over countries and sum the mortality for each country
    for i in range(len(country_mask.country)):
        print(f"Country number {i}")
        mask = country_mask.isel(country=i)
        # Sum mortality of country (based on central estimates)
        M_country = (xr.where(
            mask == 1,
            da,
            np.nan
        )).sum(dim=("lat", "lon"))
        countries.append(M_country.drop_vars("country", errors='ignore'))

    mortality_country = xr.concat(countries,
                                  dim=xr.DataArray(country_mask["country"],
                                                   dims="country",
                                                   name="country"))

    ensembles.append(mortality_country)

countries_ens = xr.concat(ensembles,
                          dim=xr.DataArray(np.arange(1, len(ensembles)+1),
                                           dims="ensemble", name="ensemble"))

countries_ens.attrs["description"] = ("Country level mean Mortality due to "
                                      "PM2.5 - scripts by A.F. Wells (2025)")
countries_ens.attrs["scenario"] = scenario
countries_ens.attrs["model"] = model
countries_ens.attrs["GBD version"] = "GBD 2023"

out_file = f"Mortality_Country_sum_{model}_{scenario}_{dates}.nc"
out_path = os.path.join(MORTALITY_DIR, out_file)

print(f"Saving country mortality to {out_path}")
countries_ens.to_netcdf(out_path)

print("All processing complete.")